# TSP: обучение на приближённом алгоритме (nearest-neighbor), инференс на разных размерностях

Этот ноутбук добавляет задачу **коммивояжёра (TSP)** к остальным алгоритмическим задачам репозитория (BFS, DFS, MST, Dijkstra, MIS). В `generate_data.py` реализованы два алгоритма генерации подсказок:

- `tsp` — эвристика **ближайшего соседа** (nearest neighbor): на каждом шаге выбирается ближайший непосещённый город от текущего. Правило локальное и сравнимое (argmin по одному скаляру на кандидата) — оно согласуется с тем, как устроена архитектура `Dnar` (жадный relax/argmin поверх скаляров в `processors.py`), поэтому именно на нём модель обучается.
- `tsp_exact` — точный алгоритм Хелда-Карпа (Held-Karp DP), даёт гарантированно кратчайший гамильтонов путь, но выбор следующего города там определяется глобальным DP по подмножествам, а не локально сравнимым числом — с такой подсказкой архитектура не выравнивается и не обучается выше ~0.35 точности по рёбрам. Поэтому здесь `tsp_exact` используется только как эталон для измерения optimality gap, а не как обучающий сигнал.

**План:**
1. Обучаем модель на подсказках **`tsp`** (ближайший сосед) на графах небольшого размера (`TRAIN_SIZE` городов).
2. Без дообучения запускаем инференс на графах бо́льшего размера:
   - для размеров `<= MAX_EXACT_TSP_SIZE` сравниваем с **точным оптимумом** (`tsp_exact`) — настоящий optimality gap;
   - для больших размеров точное решение недостижимо, поэтому сравниваем с самой эвристикой ближайшего соседа как ориентиром.

## 0. Клонирование репозитория (для запуска в Google Colab)

Ноутбуку нужны исходники репозитория (`generate_data.py`, `models.py`, `train.py`, `configs/` и т.д.). Colab стартует с пустым окружением, поэтому сначала клонируем репозиторий и переходим в его директорию; при локальном запуске ноутбука из уже склонированного репозитория эта ячейка ничего не делает.

**Важно:** ветка `REPO_BRANCH` должна содержать файлы задачи TSP (`tsp`/`tsp_exact` в `generate_data.py`, `configs/tsp.yaml`, `configs/tsp_exact.yaml`) — если изменения ещё не запушены в GitHub, запушьте их перед запуском в Colab, иначе клон не будет их содержать.

In [ ]:
import os

REPO_URL = "https://github.com/artlvruran/reasoning.git"
REPO_BRANCH = "dnar/main"
REPO_DIR = "reasoning"

if not os.path.exists("generate_data.py"):
    if not os.path.isdir(REPO_DIR):
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -r requirements.txt -q

In [ ]:
import copy
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "configs").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import torch

import generate_data
import utils
from configs import base_config
from generate_data import create_dataloader

torch.set_default_tensor_type(torch.DoubleTensor)
torch.set_num_threads(5)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("MAX_EXACT_TSP_SIZE:", generate_data.MAX_EXACT_TSP_SIZE)

## Конфигурация

Берём `configs/tsp.yaml` (алгоритм `tsp`, подсказки эвристики ближайшего соседа) и явно задаём размер обучающих графов — обучение идёт **только** на `TRAIN_SIZE` городах.

In [ ]:
config = base_config.read_config(str(repo_root / "configs" / "tsp.yaml"))

TRAIN_SIZE = 10  # размер графов при обучении (маленькая размерность)
NUM_ITERATIONS = 3000  # увеличьте для лучшего качества

config.problem_size = {
    "train": TRAIN_SIZE,
    "val": TRAIN_SIZE,
    "test": TRAIN_SIZE,
}
config.num_iterations = NUM_ITERATIONS
config.eval_each = max(NUM_ITERATIONS // 6, 1)
config.tensorboard_logs = False

print(config)

## Обучение

Переиспользуем `train.train(...)` из `train.py` — тот же цикл обучения, что и при запуске из командной строки, но результат (обученная модель) остаётся доступен в ноутбуке для последующего инференса на других размерностях.

In [ ]:
from train import train as run_training

model = run_training(config, seed=SEED)

## Инференс на разных размерностях

Веса модели не зависят от числа городов `n`, поэтому один и тот же чек-пойнт можно применить к графам произвольного размера. Проверяем, как деградирует качество по мере роста `n` относительно `TRAIN_SIZE`.

Метрики:
- **edge_accuracy / full_tour_accuracy** — точность предсказания указателей (`pointer_accuracy` / `pointer_accuracy_graph_level` из `utils.METRICS`), на уровне рёбер и на уровне всего маршрута соответственно.
- **length_ratio** — отношение длины маршрута модели к длине эталонного маршрута. Эталон зависит от размера:
  - при `size <= MAX_EXACT_TSP_SIZE` эталон — **точный оптимум** (`tsp_exact`), значит `length_ratio` это настоящий *optimality gap*;
  - при бо́льших `size` точный расчёт уже неосуществим (`O(2^n)`), поэтому эталон — маршрут **ближайшего соседа** (`tsp`), приближённый ориентир.

In [ ]:
from torch_geometric.utils import group_argsort


def tour_length_from_prediction(graph, prediction):
    edge_index = graph.edge_index
    is_selected = 1.0 * (
        group_argsort(prediction, edge_index[0], descending=True, stable=True) == 0
    )
    not_self_loop = (edge_index[0] != edge_index[1]).double()
    edge_weight = graph.scalars[:, -1, 0]
    return (is_selected * not_self_loop * edge_weight).sum()


def evaluate_tour_lengths(model, dataloader):
    total_pred_length = 0.0
    total_ref_length = 0.0
    total_graphs = 0
    for data in dataloader:
        batched_prediction, _ = model(data)
        for batch_idx, graph in enumerate(data.to_data_list()):
            batch_pred_idx = data.batch[data.edge_index[0]]
            prediction = batched_prediction[batch_pred_idx == batch_idx]
            total_pred_length += tour_length_from_prediction(graph, prediction).item()
            total_ref_length += tour_length_from_prediction(graph, graph.y).item()
            total_graphs += 1
    return total_pred_length / total_graphs, total_ref_length / total_graphs


def build_eval_dataloader(base_config_obj, algorithm, size, num_graphs, seed):
    eval_config = copy.deepcopy(base_config_obj)
    eval_config.algorithm = algorithm
    eval_config.problem_size = {"train": size, "val": size, "test": size}
    eval_config.num_samples["test"] = num_graphs
    eval_config.batch_size = min(base_config_obj.batch_size, 16)
    return create_dataloader(eval_config, "test", seed=seed, device=device)

In [ ]:
SMALL_SIZES = [8, 10, 12, 15]  # <= MAX_EXACT_TSP_SIZE: reference = exact optimum
LARGE_SIZES = [20, 30, 50, 75, 100]  # reference = nearest-neighbor heuristic
NUM_TEST_GRAPHS = 100
INFERENCE_SEED = 777

assert all(size <= generate_data.MAX_EXACT_TSP_SIZE for size in SMALL_SIZES)
assert all(size > generate_data.MAX_EXACT_TSP_SIZE for size in LARGE_SIZES)

model.eval()
results = []

with torch.no_grad():
    for size, algorithm, reference in [
        *((size, "tsp_exact", "exact") for size in SMALL_SIZES),
        *((size, "tsp", "nn") for size in LARGE_SIZES),
    ]:
        test_data = build_eval_dataloader(
            config, algorithm, size, NUM_TEST_GRAPHS, INFERENCE_SEED
        )

        scores = utils.evaluate(model, test_data, utils.METRICS["pointer"])
        pred_length, ref_length = evaluate_tour_lengths(model, test_data)

        results.append(
            {
                "size": size,
                "reference": reference,
                "edge_accuracy": scores["pointer_accuracy"],
                "full_tour_accuracy": scores["pointer_accuracy_graph_level"],
                "length_ratio": pred_length / ref_length,
            }
        )
        print(
            f"n={size:4d}  ref={reference:5s}  "
            f"edge_acc={scores['pointer_accuracy']:.3f}  "
            f"full_tour_acc={scores['pointer_accuracy_graph_level']:.3f}  "
            f"length_ratio={pred_length / ref_length:.3f}"
        )

model.train()

## Визуализация

In [ ]:
import matplotlib.pyplot as plt

all_sizes = [r["size"] for r in results]
edge_acc = [r["edge_accuracy"] for r in results]
full_acc = [r["full_tour_accuracy"] for r in results]
exact_results = [r for r in results if r["reference"] == "exact"]
nn_results = [r for r in results if r["reference"] == "nn"]

BLUE = "#2a78d6"
ORANGE = "#eb6834"
MUTED = "#898781"
GRID = "#e1e0d9"
INK = "#0b0b0b"
SURFACE = "#fcfcfb"

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), facecolor=SURFACE)

ax = axes[0]
ax.set_facecolor(SURFACE)
ax.plot(all_sizes, edge_acc, color=BLUE, linewidth=2, marker="o", markersize=6, label="edge accuracy")
ax.plot(all_sizes, full_acc, color=ORANGE, linewidth=2, marker="o", markersize=6, label="full-tour accuracy")
ax.axvline(TRAIN_SIZE, color=MUTED, linestyle="--", linewidth=1)
ax.text(TRAIN_SIZE, 1.03, "train size", color=MUTED, fontsize=9, ha="center", va="bottom")
ax.axvline(generate_data.MAX_EXACT_TSP_SIZE, color=MUTED, linestyle=":", linewidth=1)
ax.text(
    generate_data.MAX_EXACT_TSP_SIZE, 0.03, "exact reference ends",
    color=MUTED, fontsize=8, ha="center", va="bottom", rotation=90,
)
ax.set_xlabel("problem size (number of cities)", color=INK)
ax.set_ylabel("accuracy", color=INK)
ax.set_title("Pointer accuracy vs. problem size", color=INK)
ax.set_ylim(0, 1.12)
ax.grid(True, color=GRID, linewidth=0.8)
ax.tick_params(colors=MUTED)
for spine in ax.spines.values():
    spine.set_color(GRID)
ax.legend(frameon=False, labelcolor=INK, loc="lower left")

ax2 = axes[1]
ax2.set_facecolor(SURFACE)
if exact_results:
    ax2.plot(
        [r["size"] for r in exact_results],
        [r["length_ratio"] for r in exact_results],
        color=BLUE, linewidth=2, marker="o", markersize=6,
        label="vs. exact optimum",
    )
if nn_results:
    ax2.plot(
        [r["size"] for r in nn_results],
        [r["length_ratio"] for r in nn_results],
        color=ORANGE, linewidth=2, marker="o", markersize=6,
        label="vs. nearest-neighbor",
    )
ax2.axhline(1.0, color=MUTED, linestyle="--", linewidth=1)
ax2.axvline(TRAIN_SIZE, color=MUTED, linestyle="--", linewidth=1)
ax2.set_xlabel("problem size (number of cities)", color=INK)
ax2.set_ylabel("predicted / reference tour length", color=INK)
ax2.set_title("Tour length ratio vs. problem size", color=INK)
ax2.grid(True, color=GRID, linewidth=0.8)
ax2.tick_params(colors=MUTED)
for spine in ax2.spines.values():
    spine.set_color(GRID)
ax2.legend(frameon=False, labelcolor=INK)

fig.tight_layout()
plt.show()

## Итоги

- Модель обучена на графах TSP с `TRAIN_SIZE` городами, используя подсказки эвристики **ближайшего соседа** (`tsp`) — локально сравнимое правило, согласованное с архитектурой `Dnar`.
- Инференс запускается на графах бо́льшего размера **без дообучения** — сеть работает одинаково для любого `n`, так как её параметры не зависят от размера графа.
- В диапазоне `size <= MAX_EXACT_TSP_SIZE` `length_ratio` считается против **точного оптимума** (`tsp_exact`) — это настоящий optimality gap самой модели, а не только её обучающего сигнала. Дальше точное решение экспоненциально дорого, поэтому ориентиром служит сама эвристика ближайшего соседа — грубая проверка того, что модель продолжает строить разумные маршруты на бо́льших графах.
- Точность и качество маршрута ожидаемо деградируют по мере роста разрыва между `n` при обучении и при инференсе — типичный эффект в neural algorithmic reasoning. Для улучшения обобщения можно увеличить `NUM_ITERATIONS`, `TRAIN_SIZE` или число обучающих графов (`config.num_samples["train"]`).

Результаты сохранены в переменной `results` (список словарей с полем `reference`, указывающим эталон сравнения) для дальнейшего анализа.